# CodeMix-T — Phase 3: Training Loop

Covers:
1. Mount drive & reload architecture
2. Dataset class & DataLoader
3. Label-smoothed cross-entropy loss
4. Warmup + inverse-sqrt LR scheduler (Vaswani schedule)
5. Training step & validation step
6. Full training loop with checkpointing
7. WandB experiment tracking
8. Resume from checkpoint

## Cell 1 — Setup & Mount Drive

In [ ]:
!pip install -q sentencepiece wandb

import torch, math, time, json, os
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import sentencepiece as spm
import wandb

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/CodeMixT'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Cell 2 — Paste Architecture (copy all classes from Phase 2 here)

In [ ]:
# ── PASTE FULL ARCHITECTURE CODE FROM PHASE 2 HERE ──
# Copy everything from Cell 2 (CodeMixTConfig) through Cell 8 (masks)
# and the CodeMixT class from Cell 9.
#
# Then load config and model:

with open(f'{DRIVE_DIR}/model/config.json') as f:
    cfg_dict = json.load(f)

from dataclasses import dataclass
# (assuming CodeMixTConfig is pasted above)
cfg = CodeMixTConfig(**cfg_dict)

model = CodeMixT(cfg).to(device)
model.load_state_dict(torch.load(f'{DRIVE_DIR}/model/codemix_t_init.pt', map_location=device))
print(f'Model loaded: {model.count_parameters()/1e6:.1f}M params')

## Cell 3 — Dataset Class

In [ ]:
class CodeMixDataset(Dataset):
    """
    Loads pre-tokenized JSONL files from Phase 1.
    Each record has: src_ids, tgt_ids, source_lang_ids
    """
    def __init__(self, jsonl_path: str, max_src_len: int = 128, max_tgt_len: int = 128):
        self.samples = []
        self.max_src = max_src_len
        self.max_tgt = max_tgt_len

        with open(jsonl_path) as f:
            for line in f:
                rec = json.loads(line)
                src = rec['src_ids']
                tgt = rec['tgt_ids']
                lid = list(map(int, rec['source_lang_ids'].split()))

                # Align lang_ids with tokenized src (best effort)
                if len(lid) < len(src):
                    lid = lid + [0] * (len(src) - len(lid))
                lid = lid[:len(src)]

                if 3 <= len(src) <= max_src_len and 3 <= len(tgt) <= max_tgt_len:
                    self.samples.append((src, tgt, lid))

        print(f'Loaded {len(self.samples):,} samples from {Path(jsonl_path).name}')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        src, tgt, lid = self.samples[idx]
        return (
            torch.tensor(src, dtype=torch.long),
            torch.tensor(tgt, dtype=torch.long),
            torch.tensor(lid, dtype=torch.long)
        )


def collate_fn(batch, pad_id=0):
    """Pad sequences in a batch to the same length."""
    srcs, tgts, lids = zip(*batch)

    src_lens = [s.size(0) for s in srcs]
    tgt_lens = [t.size(0) for t in tgts]
    max_src  = max(src_lens)
    max_tgt  = max(tgt_lens)

    src_pad  = torch.full((len(srcs), max_src), pad_id, dtype=torch.long)
    tgt_pad  = torch.full((len(tgts), max_tgt), pad_id, dtype=torch.long)
    lid_pad  = torch.zeros(len(lids), max_src, dtype=torch.long)

    for i, (s, t, l) in enumerate(zip(srcs, tgts, lids)):
        src_pad[i, :s.size(0)] = s
        tgt_pad[i, :t.size(0)] = t
        lid_pad[i, :l.size(0)] = l

    return src_pad, tgt_pad, lid_pad


# Load datasets
train_ds = CodeMixDataset(f'{DRIVE_DIR}/data_final/train.jsonl')
val_ds   = CodeMixDataset(f'{DRIVE_DIR}/data_final/val.jsonl')

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=lambda b: collate_fn(b, cfg.pad_id), num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=lambda b: collate_fn(b, cfg.pad_id), num_workers=2)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## Cell 4 — Label-Smoothed Cross Entropy Loss

In [ ]:
class LabelSmoothingLoss(nn.Module):
    """
    Cross-entropy with label smoothing.
    Instead of hard 0/1 targets, smooth to (eps/V) for wrong classes
    and (1 - eps + eps/V) for the correct class.
    Prevents overconfidence and improves generalisation (Szegedy et al. 2016).
    eps=0.1 is standard for translation tasks.
    """
    def __init__(self, vocab_size: int, pad_id: int, smoothing: float = 0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_id     = pad_id
        self.smoothing  = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits:  [B * T, vocab_size]
        # targets: [B * T]
        B = logits.size(0)
        log_probs = F.log_softmax(logits, dim=-1)

        # Smoothed target distribution
        smooth_val = self.smoothing / (self.vocab_size - 2)  # -2 for pad and true class
        with torch.no_grad():
            dist = torch.full_like(log_probs, smooth_val)
            dist.scatter_(1, targets.unsqueeze(1), self.confidence)
            dist[:, self.pad_id] = 0  # Never predict PAD
            mask = (targets == self.pad_id)
            dist[mask] = 0            # Zero out pad positions

        loss = -(dist * log_probs).sum(dim=-1)
        non_pad = (~mask).sum()
        return loss.sum() / non_pad.clamp(min=1)


criterion = LabelSmoothingLoss(cfg.vocab_size, cfg.pad_id, smoothing=0.1)
print('LabelSmoothingLoss defined.')

## Cell 5 — Optimizer & LR Scheduler (Vaswani et al. warmup schedule)

In [ ]:
class WarmupInvSqrtScheduler:
    """
    Learning rate schedule from 'Attention Is All You Need'.
    lr = d_model^-0.5 * min(step^-0.5, step * warmup_steps^-1.5)

    - Linearly increases LR during warmup
    - Decreases proportionally to inverse sqrt of step after warmup
    - warmup_steps=4000 is the standard value
    """
    def __init__(self, optimizer, d_model: int, warmup_steps: int = 4000):
        self.optimizer    = optimizer
        self.d_model      = d_model
        self.warmup_steps = warmup_steps
        self.step_num     = 0

    def step(self):
        self.step_num += 1
        lr = self._get_lr()
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr
        return lr

    def _get_lr(self):
        s = self.step_num
        w = self.warmup_steps
        return (self.d_model ** -0.5) * min(s ** -0.5, s * w ** -1.5)

    def state_dict(self):
        return {'step_num': self.step_num}

    def load_state_dict(self, sd):
        self.step_num = sd['step_num']


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1.0,             # Scheduler controls the actual LR
    betas=(0.9, 0.98),
    eps=1e-9
)
scheduler = WarmupInvSqrtScheduler(optimizer, d_model=cfg.d_model, warmup_steps=4000)

print('Optimizer and scheduler ready.')

# Visualise the LR schedule
steps = list(range(1, 20001))
_sched = WarmupInvSqrtScheduler(torch.optim.Adam([torch.zeros(1)]), cfg.d_model, 4000)
lrs = []
for s in steps:
    _sched.step_num = s
    lrs.append(_sched._get_lr())
peak_step = lrs.index(max(lrs)) + 1
print(f'Peak LR = {max(lrs):.6f} at step {peak_step}')
print(f'LR at step 1000  = {lrs[999]:.6f}')
print(f'LR at step 10000 = {lrs[9999]:.6f}')

## Cell 6 — Training & Validation Steps

In [ ]:
def train_step(model, batch, optimizer, scheduler, criterion, device, grad_clip=1.0):
    """
    Single training step.
    Uses teacher forcing: feed ground-truth target tokens as decoder input.
    """
    model.train()
    src_ids, tgt_ids, lang_ids = [x.to(device) for x in batch]

    # Teacher forcing:
    # Decoder input  = tgt[:-1]  (all tokens except last)
    # Decoder target = tgt[1:]   (all tokens except first BOS)
    dec_input  = tgt_ids[:, :-1]
    dec_target = tgt_ids[:, 1:]

    # Forward pass
    logits = model(src_ids, lang_ids, dec_input)  # [B, T-1, vocab]

    # Compute loss
    B, T, V = logits.shape
    loss = criterion(logits.reshape(B*T, V), dec_target.reshape(B*T))

    # Backprop
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    lr = scheduler.step()

    return loss.item(), lr


@torch.no_grad()
def val_step(model, val_loader, criterion, device):
    """Full validation pass — returns avg loss and perplexity."""
    model.eval()
    total_loss, n_batches = 0.0, 0

    for batch in val_loader:
        src_ids, tgt_ids, lang_ids = [x.to(device) for x in batch]
        dec_input  = tgt_ids[:, :-1]
        dec_target = tgt_ids[:, 1:]

        logits = model(src_ids, lang_ids, dec_input)
        B, T, V = logits.shape
        loss = criterion(logits.reshape(B*T, V), dec_target.reshape(B*T))
        total_loss += loss.item()
        n_batches  += 1

    avg_loss   = total_loss / n_batches
    perplexity = math.exp(min(avg_loss, 20))  # Clip to avoid overflow
    return avg_loss, perplexity


print('train_step and val_step defined.')

## Cell 7 — Checkpoint Utilities

In [ ]:
CKPT_DIR = f'{DRIVE_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)


def save_checkpoint(model, optimizer, scheduler, epoch, step, val_loss, tag='latest'):
    path = f'{CKPT_DIR}/codemix_t_{tag}.pt'
    torch.save({
        'epoch':      epoch,
        'step':       step,
        'val_loss':   val_loss,
        'model':      model.state_dict(),
        'optimizer':  optimizer.state_dict(),
        'scheduler':  scheduler.state_dict(),
    }, path)
    print(f'Checkpoint saved: {path}  (val_loss={val_loss:.4f})')


def load_checkpoint(model, optimizer, scheduler, tag='latest'):
    path = f'{CKPT_DIR}/codemix_t_{tag}.pt'
    if not os.path.exists(path):
        print(f'No checkpoint found at {path}')
        return 0, 0, float('inf')

    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    print(f'Loaded checkpoint: epoch={ckpt["epoch"]}, step={ckpt["step"]}, val_loss={ckpt["val_loss"]:.4f}')
    return ckpt['epoch'], ckpt['step'], ckpt['val_loss']


print('Checkpoint utilities ready.')

## Cell 8 — WandB Initialisation

In [ ]:
# Create a free WandB account at wandb.ai, then paste your API key when prompted
wandb.login()

run = wandb.init(
    project='codemix-t',
    name='base-run-1',
    config={
        'd_model':        cfg.d_model,
        'd_ff':           cfg.d_ff,
        'num_heads':      cfg.num_heads,
        'enc_layers':     cfg.num_enc_layers,
        'dec_layers':     cfg.num_dec_layers,
        'vocab_size':     cfg.vocab_size,
        'lang_embed_dim': cfg.lang_embed_dim,
        'batch_size':     BATCH_SIZE,
        'warmup_steps':   4000,
        'label_smoothing': 0.1,
        'grad_clip':      1.0,
        'total_params':   model.count_parameters(),
    }
)
print(f'WandB run: {run.name}')

## Cell 9 — Full Training Loop

In [ ]:
NUM_EPOCHS      = 30
VAL_EVERY       = 1          # Validate every N epochs
SAVE_EVERY      = 1          # Save checkpoint every N epochs
LOG_EVERY       = 50         # Log to WandB every N steps
PATIENCE        = 5          # Early stopping patience

# Optionally resume from checkpoint
start_epoch, global_step, best_val_loss = load_checkpoint(model, optimizer, scheduler)

no_improve = 0

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for batch in pbar:
        loss, lr = train_step(model, batch, optimizer, scheduler, criterion, device)
        epoch_loss  += loss
        global_step += 1

        pbar.set_postfix({'loss': f'{loss:.4f}', 'lr': f'{lr:.2e}'})

        if global_step % LOG_EVERY == 0:
            wandb.log({'train/loss': loss, 'train/lr': lr, 'step': global_step})

    avg_train_loss = epoch_loss / len(train_loader)
    epoch_time     = time.time() - t0

    # Validation
    if (epoch + 1) % VAL_EVERY == 0:
        val_loss, val_ppl = val_step(model, val_loader, criterion, device)

        print(f'Epoch {epoch+1:3d} | '
              f'train_loss={avg_train_loss:.4f} | '
              f'val_loss={val_loss:.4f} | '
              f'val_ppl={val_ppl:.2f} | '
              f'time={epoch_time:.0f}s')

        wandb.log({
            'epoch':      epoch + 1,
            'train/epoch_loss': avg_train_loss,
            'val/loss':   val_loss,
            'val/ppl':    val_ppl,
        })

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(model, optimizer, scheduler, epoch+1, global_step, val_loss, tag='best')
            no_improve = 0
        else:
            no_improve += 1
            print(f'  No improvement for {no_improve}/{PATIENCE} epochs')

        # Early stopping
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}')
            break

    # Periodic checkpoint
    if (epoch + 1) % SAVE_EVERY == 0:
        save_checkpoint(model, optimizer, scheduler, epoch+1, global_step, val_loss, tag='latest')

print(f'\nTraining complete. Best val_loss = {best_val_loss:.4f}')
wandb.finish()